In [1]:
import sys
import yaml
from pathlib import Path

import numpy as np 
import torch 
from tqdm import tqdm 
from einops import rearrange 

from eb_jepa.datasets.utils import init_data
from eb_jepa.training_utils import load_config
from eb_jepa.datasets.two_rooms.env import DotWall
from eb_jepa.vis_utils import create_comparison_gif, show_images

In [2]:
PKG_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "cfgs" / "train.yaml").exists()
)
sys.path.insert(0, str(PKG_ROOT))

from planning import GCAgent
from builders import build_model


TRAIN_CFG_PATH = PKG_ROOT / "cfgs" / "train.yaml"
EVAL_CFG_PATH  = PKG_ROOT / "cfgs" / "eval.yaml"

### Train cfg

In [3]:
cfg = load_config(TRAIN_CFG_PATH)  # in order to use dot notation 

loader, val_loader, data_config = init_data(
    env_name=cfg.data.env_name, cfg_data=dict(cfg.data)
)

[INFO    ][2026-09-03 11:08:21][eb_jepa.training_utils][load_config              ] Loaded config from /Users/hawardizayee/Desktop/AMI/eb_jepa/examples/my_ac_video_jepa/cfgs/train.yaml


/Users/hawardizayee/Desktop/AMI/eb_jepa/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 10 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


### Eval cfg

In [4]:
with open(EVAL_CFG_PATH, "r") as f:
    eval_cfg_dict = yaml.safe_load(f)

# loader, val_loader, env_config
_, _, env_config = init_data(
    env_name="two_rooms",
    cfg_data=dict(eval_cfg_dict.get("data",{}))
)

cfg_eval_env = eval_cfg_dict.get("env")

def env_creator():
    cfg_eval_env = eval_cfg_dict.get("env")
    return DotWall(
        config=env_config,
        **cfg_eval_env
    )

### JEPA 

In [5]:
jepa, xy_prober = build_model(
    cfg, 
    data_config=data_config, 
    normalizer=loader.dataset.normalizer
    )

In [6]:
PLAN_CFG_PATH = PKG_ROOT / "cfgs" / "planning_mppi.yaml"

with open(PLAN_CFG_PATH, "r") as f:
    plan_cfg = yaml.safe_load(f)

In [7]:
from omegaconf import OmegaConf

plan_cfg = OmegaConf.create(plan_cfg)

def env_creator():
    cfg_eval_env = eval_cfg_dict.get("env")
    return DotWall(
        config = env_config,
        **cfg_eval_env
    )

plan_cfg.planner.planner_name

'mppi'

In [8]:
env = env_creator()

agent = GCAgent(
    jepa,
    action_dim=2,
    plan_cfg=plan_cfg,
    normalizer=env.normalizer,
    loc_prober=xy_prober,
    env=env
)


In [9]:
env.reset()
obs_init, done, done, truncted, info = env.step(np.zeros(2,))
obs_tensor = (
    env.normalizer.normalize_state(
        obs_init.detach().clone().to(dtype=torch.float32, device=agent.device)
    )
    .unsqueeze(0)
    .unsqueeze(2)
)   # Unsqueeze the batch and time dimensions : C H W -> 1 C 1 H W

print(obs_tensor.shape)

torch.Size([1, 2, 1, 65, 65])


In [10]:


steps_left = env.n_allowed_steps    # 200 
plan_length = min(agent.planner.plan_length, steps_left)    #90

mean = torch.zeros(plan_length, agent.planner.action_dim)                                   # (90, 2)
std = agent.planner.max_std * torch.ones(plan_length, agent.planner.action_dim)             # (90, 2)
actions = torch.empty(
    plan_length, 
    agent.planner.num_samples,
    agent.planner.action_dim
    )      # (90, 200, 2)

losses = []
elite_means = []
elite_stds = []


for _ in range(agent.planner.n_iters):  # 20 
    actions[:, :] = mean.unsqueeze(1) + std.unsqueeze(1) * torch.randn(
        plan_length,                # 90    
        agent.planner.num_samples,   # 200
        agent.planner.action_dim    # 2 
    )   # (T=90, B=200, A=2) 
    
    # Compute costs 
    # cost function does the two following line 
    # predicted_encs = self.unroll(obs_init, actions)
    # self.objective(predicted_encs)
    cost = agent.planner.cost_function(         
        rearrange(actions, "t b a -> b a t"), obs_tensor  # actions, obs_tensor
    ).unsqueeze(1)
    losses.append(cost.min().item())

    # Get elite actions 
    elite_idxs = torch.topk(-cost.squeeze(1), agent.planner.num_elites, dim=0).indices
    print(f'idx : {elite_idxs}')
    elite_loss, elite_actions = cost[elite_idxs], actions[:, elite_idxs]

    # record statistics 
    elite_means.append(elite_loss.mean().item())
    elite_stds.append(elite_loss.std().item())

    # Update parameters 
    min_cost = cost.min(0)[0]   # shape = (1,)
    score = torch.exp(
        agent.planner.temperature * (min_cost - elite_loss[:, 0])
    )
    score /= score.sum(0)
    mean = torch.sum(
        score.unsqueeze(0).unsqueeze(2) * elite_actions, dim=1 
    ) / (score.sum(0) + 1e-9)

    std = torch.sqrt(
        torch.sum(
            score.unsqueeze(0).unsqueeze(2) 
            * (elite_actions - mean.unsqueeze(1)) **2,
            dim =1
        )
        / (score.sum(0) + 1e-9)
    )

TypeError: 'NoneType' object is not callable